# Notebook 03 — TabSyn Synthetic Generation
Generate **720,000 synthetic sessions** using TabSyn (ICLR 2023).

TabSyn uses a two-stage pipeline:
1. **VAE** — learns a continuous latent space from tabular rows
2. **Score-based diffusion** — generates in that latent space

This preserves joint feature correlations (EPSS stays high when CVSS is high)
better than TabDDPM or CTGAN.

**Run on:** College system (RTX 4500 Ada, 24 GB VRAM) or laptop (RTX 3050, 4 GB)  
**Expected time:** ~4h (college) / ~6-8h (laptop)

**TabSyn is a cloned repo** at `../tabsyn/`, not a pip package. Training is
run via `python main.py` from the TabSyn directory. This notebook prepares the
data in TabSyn's expected format, launches training, then post-processes output.

In [ ]:
import sys, logging, warnings, json, os, subprocess, time, traceback
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger()

ROOT       = Path("..").resolve()
DATA_PROC  = ROOT / "data" / "processed"
DATA_SYNTH = ROOT / "data" / "synthetic"
DATA_SYNTH.mkdir(parents=True, exist_ok=True)
TABSYN_ROOT = (ROOT / ".." / "tabsyn").resolve()
STATUS_FILE = DATA_SYNTH / "tabsyn_status.txt"
sys.path.insert(0, str(ROOT))

X_real = np.load(DATA_PROC / "X_real.npy")
y_real = np.load(DATA_PROC / "y_real.npy")
print(f"Real data: X={X_real.shape}  y={y_real.shape}")
print(f"Classes present: {len(np.unique(y_real))}/45")
print(f"TabSyn repo: {TABSYN_ROOT}")
print(f"Status file: {STATUS_FILE}")

## 3.1 Prepare TabSyn Input
TabSyn expects npy files in `tabsyn/data/{dataname}/`:
- `X_num_train.npy`, `X_num_test.npy` — numerical features (float32)
- `y_train.npy`, `y_test.npy` — labels
- `info.json` — dataset metadata

Our dataset is 100% numerical (128 features, no categoricals).

In [ ]:
from configs.schema import FEAT_NAMES, IDX_TO_LABEL
from sklearn.utils import resample

DATANAME = "honeypot_sessions"
TABSYN_DATA_DIR = TABSYN_ROOT / "data" / DATANAME
TABSYN_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Oversample rare classes so TabSyn sees balanced training data
df_input = pd.DataFrame(X_real, columns=FEAT_NAMES)
df_input["micro_state"] = y_real

frames = []
target_per_class = max(500, len(df_input) // 45)
for cls in df_input["micro_state"].unique():
    sub = df_input[df_input["micro_state"] == cls]
    if len(sub) < target_per_class:
        sub = resample(sub, replace=True, n_samples=target_per_class, random_state=42)
    frames.append(sub)
df_balanced = pd.concat(frames).sample(frac=1, random_state=42).reset_index(drop=True)

# Train/test split (90/10)
X_all = df_balanced[FEAT_NAMES].values.astype(np.float32)
y_all = df_balanced["micro_state"].values.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.1, random_state=42, stratify=y_all)

# Save in TabSyn's expected format
np.save(TABSYN_DATA_DIR / "X_num_train.npy", X_train)
np.save(TABSYN_DATA_DIR / "X_num_test.npy",  X_test)
np.save(TABSYN_DATA_DIR / "y_train.npy",     y_train)
np.save(TABSYN_DATA_DIR / "y_test.npy",      y_test)

# TabSyn requires X_cat files to exist even with no categorical features
X_cat_train = np.empty((len(X_train), 0), dtype=str)
X_cat_test  = np.empty((len(X_test),  0), dtype=str)
np.save(TABSYN_DATA_DIR / "X_cat_train.npy", X_cat_train)
np.save(TABSYN_DATA_DIR / "X_cat_test.npy",  X_cat_test)

# Save train/test CSVs (TabSyn uses these for evaluation)
cols_all = FEAT_NAMES + ["micro_state"]
df_train = pd.DataFrame(np.column_stack([X_train, y_train]), columns=cols_all)
df_test  = pd.DataFrame(np.column_stack([X_test,  y_test]),  columns=cols_all)
df_train.to_csv(TABSYN_DATA_DIR / "train.csv", index=False)
df_test.to_csv(TABSYN_DATA_DIR / "test.csv",   index=False)

# Create info.json
info = {
    "name":           DATANAME,
    "task_type":      "multiclass",
    "header":         0,
    "column_names":   cols_all,
    "num_col_idx":    list(range(128)),
    "cat_col_idx":    [],
    "target_col_idx": [128],
    "n_classes":      45,
    "train_num":      len(X_train),
    "test_num":       len(X_test),
}

with open(TABSYN_DATA_DIR / "info.json", "w") as f:
    json.dump(info, f, indent=2)

# Create synthetic output directory
synth_out = TABSYN_ROOT / "synthetic" / DATANAME
synth_out.mkdir(parents=True, exist_ok=True)
df_train.to_csv(synth_out / "real.csv", index=False)
df_test.to_csv(synth_out / "test.csv", index=False)

print(f"TabSyn data prepared at: {TABSYN_DATA_DIR}")
print(f"  X_num_train: {X_train.shape}")
print(f"  X_num_test:  {X_test.shape}")
print(f"  y_train: {y_train.shape} ({len(np.unique(y_train))} classes)")
print(f"  X_cat files: dummy (0 cols) — required by TabSyn")
print(f"  info.json written")

## 3.2 Train TabSyn (VAE + Diffusion)
Two-stage training run from the `tabsyn/` directory:
1. `--method vae` trains the VAE encoder/decoder
2. `--method tabsyn --mode train` trains diffusion in latent space

Crash-safe: writes status to `data/synthetic/tabsyn_status.txt` so you can
check in the morning whether it succeeded or where it failed.

In [ ]:
import torch

gpu_mem = torch.cuda.get_device_properties(0).total_mem if torch.cuda.is_available() else 0

# VAE uses attention → VRAM-hungry. Diffusion uses MLP → more headroom.
# RTX 3050 4GB: batch 512 for VAE, 4096 for diffusion
# RTX 4500 24GB: batch 4096 for both
if gpu_mem > 8e9:
    vae_batch = 4096
    diff_batch = 4096
else:
    vae_batch = 512
    diff_batch = 4096

vae_epochs = 500
diff_epochs = 2000

def write_status(msg):
    with open(STATUS_FILE, "w") as f:
        f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} | {msg}\n")

def run_tabsyn_step(method, mode, label, batch_override=None, extra_args=None):
    """Run one TabSyn CLI step with crash-safe status logging."""
    bs = batch_override or (vae_batch if method == "vae" else diff_batch)
    epochs = vae_epochs if method == "vae" else diff_epochs
    cmd = [
        sys.executable, "main.py",
        "--dataname", DATANAME,
        "--method",   method,
        "--mode",     mode,
        "--epochs",   str(epochs),
        "--training_batch_size", str(bs),
    ]
    if extra_args:
        cmd.extend(extra_args)

    write_status(f"RUNNING: {label} | cmd: {' '.join(cmd)}")
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"  cmd: {' '.join(cmd)}")
    print(f"  cwd: {TABSYN_ROOT}")
    print(f"{'='*60}\n")

    t0 = time.time()
    try:
        result = subprocess.run(
            cmd, cwd=str(TABSYN_ROOT),
            capture_output=False, text=True, timeout=43200  # 12h timeout
        )
        elapsed = time.time() - t0
        if result.returncode != 0:
            write_status(f"FAILED: {label} | exit code {result.returncode} | {elapsed/60:.1f} min")
            raise RuntimeError(f"{label} failed with exit code {result.returncode}")
        write_status(f"SUCCESS: {label} | {elapsed/60:.1f} min")
        print(f"\n{label} completed in {elapsed/60:.1f} min")
    except Exception as e:
        elapsed = time.time() - t0
        write_status(f"CRASHED: {label} | {elapsed/60:.1f} min | {traceback.format_exc()}")
        raise

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {gpu_mem/1e9:.1f} GB")
print(f"VAE batch: {vae_batch}  |  Diffusion batch: {diff_batch}")
print(f"VAE epochs: {vae_epochs}  |  Diffusion epochs: {diff_epochs}")
print(f"\nStatus file: {STATUS_FILE}")
print(f"Check this file in the morning to see if training succeeded.")
print()

# ── Stage 1: VAE ──
run_tabsyn_step("vae", "train", "Stage 1/3: VAE training")

# ── Stage 2: Diffusion ──
run_tabsyn_step("tabsyn", "train", "Stage 2/3: Diffusion training")

write_status("ALL TRAINING COMPLETE")
print("\nAll training stages complete!")

## 3.3 Sample 720,000 Synthetic Sessions
Sampling is much faster than training (~25 min on RTX 3050).

In [ ]:
N_SAMPLE = 720_000
SAVE_PATH = str(DATA_SYNTH / "tabsyn_raw.csv")

run_tabsyn_step(
    "tabsyn", "sample", "Stage 3/3: Sampling 720k sessions",
    extra_args=["--save_path", SAVE_PATH, "--num-samples", str(N_SAMPLE)]
)

# Load the generated samples
tabsyn_csv = Path(SAVE_PATH)
if tabsyn_csv.exists():
    df_tabsyn = pd.read_csv(tabsyn_csv)
    print(f"Loaded TabSyn samples: {len(df_tabsyn):,} rows  {len(df_tabsyn.columns)} cols")
else:
    # TabSyn may save to its own synthetic/ dir instead
    alt_path = TABSYN_ROOT / "synthetic" / DATANAME / "tabsyn.csv"
    if alt_path.exists():
        df_tabsyn = pd.read_csv(alt_path)
        print(f"Loaded from TabSyn default path: {len(df_tabsyn):,} rows")
    else:
        raise FileNotFoundError(
            f"TabSyn output not found at {tabsyn_csv} or {alt_path}"
        )

## 3.4 Post-Process TabSyn Output
- Round micro_state to valid class index
- Add session metadata + kill-chain sequences
- Inject EPSS temporal drift for CVE-associated sessions
- Save feature matrix + metadata

In [ ]:
import hashlib
from configs.schema import FEAT_NAMES, LABEL_TO_IDX, IDX_TO_LABEL
from src.generators.kill_chain_simulator import filter_invalid_sequences
from src.generators.epss_drift import inject_multi_cve_drift

# ── 1. Align columns to schema ──
# TabSyn outputs columns in the order it was trained on
if "micro_state" not in df_tabsyn.columns:
    df_tabsyn.columns = FEAT_NAMES + ["micro_state"]

# Round micro_state to nearest valid class index
df_tabsyn["micro_state"] = df_tabsyn["micro_state"].round().clip(0, 44).astype(int)
df_tabsyn["micro_state_label"] = df_tabsyn["micro_state"].map(IDX_TO_LABEL)

# ── 2. Add session metadata ──
df_tabsyn["session_id"] = [
    "ts_" + hashlib.md5(str(i).encode()).hexdigest()[:8]
    for i in range(len(df_tabsyn))
]
df_tabsyn["source"] = "tabsyn_synthetic"
df_tabsyn["session_day"] = np.random.randint(0, 90, size=len(df_tabsyn))
df_tabsyn["associated_cve"] = ""

# ── 3. Kill-chain sequence (single-label for TabSyn rows) ──
df_tabsyn["micro_state_sequence"] = df_tabsyn["micro_state_label"]
print(f"Before kill-chain filter: {len(df_tabsyn):,} sessions")
print(f"After  kill-chain filter: {len(df_tabsyn):,} sessions (single-label always passes)")

# ── 4. Inject EPSS temporal drift for CVE-associated sessions ──
cve_schedule = [
    {"cve_id": "CVE-2023-44487",   "kev_day": 10},
    {"cve_id": "CVE-2024-3400",    "kev_day": 45},
    {"cve_id": "CVE-2023-20198",   "kev_day": 25},
    {"cve_id": "CVE-2023-46805",   "kev_day": 55},
    {"cve_id": "CVE-2024-21762",   "kev_day": 70},
]
high_risk_mask = df_tabsyn["micro_state"].isin(range(23, 45))
for i, sch in enumerate(cve_schedule):
    chunk_mask = high_risk_mask & (df_tabsyn.index % len(cve_schedule) == i)
    df_tabsyn.loc[chunk_mask, "associated_cve"] = sch["cve_id"]

df_tabsyn = inject_multi_cve_drift(df_tabsyn, cve_schedule, seed=42)

# ── 5. Extract feature matrix and save ──
X_tabsyn = df_tabsyn[FEAT_NAMES].values.astype(np.float32)
y_tabsyn = df_tabsyn["micro_state"].values.astype(np.int64)
X_tabsyn = np.nan_to_num(X_tabsyn, nan=0.0, posinf=1e4, neginf=-1e4)

print(f"\nTabSyn feature matrix: {X_tabsyn.shape}")
print(f"Class distribution (first 10):")
from collections import Counter
counts = Counter(y_tabsyn.tolist())
for cls_id in sorted(counts)[:10]:
    print(f"  [{cls_id:2d}] {IDX_TO_LABEL[cls_id]:<30} {counts[cls_id]:>6,}")

np.save(DATA_SYNTH / "X_tabsyn.npy", X_tabsyn)
np.save(DATA_SYNTH / "y_tabsyn.npy", y_tabsyn)
df_tabsyn[["session_id","micro_state","micro_state_label","source",
           "session_day","associated_cve"]].to_parquet(
    DATA_SYNTH / "tabsyn_meta.parquet", index=False)
print(f"\nSaved X_tabsyn.npy {X_tabsyn.shape}  y_tabsyn.npy {y_tabsyn.shape}")
write_status(f"POST-PROCESSING COMPLETE | {len(df_tabsyn):,} sessions saved")

## 3.5 Quick Quality Check on TabSyn Output
Adversarial AUC: can XGBoost tell real from synthetic? (target: < 0.60)
Wasserstein distance: per-group distributional fidelity (target: mean < 0.5)

In [ ]:
from src.validators.quality_checks import adversarial_auc, wasserstein_by_group

print("Running adversarial AUC (XGBoost real vs. TabSyn synthetic)...")
adv = adversarial_auc(X_real, X_tabsyn[:len(X_real)], n_sample=10_000)
print(f"  Adversarial AUC: {adv['adversarial_auc']}  -> {adv['verdict']}")

print("\nWasserstein distance by feature group:")
n = min(len(X_real), len(X_tabsyn))
wass = wasserstein_by_group(X_real[:n], X_tabsyn[:n])
for grp, res in wass.items():
    status = "PASS" if res["passed"] else "WARN"
    print(f"  {status} {grp:<22} mean_W={res['mean_W']:.4f}  max_W={res['max_W']:.4f}")

print()
if adv["adversarial_auc"] > 0.65:
    print("WARNING: AUC too high -- consider:")
    print("   1. Increasing TabSyn training epochs (try 3000)")
    print("   2. Checking for feature scaling issues in the leaky features")
    print(f"   Leaky features: {adv['top10_leaky_feature_indices'][:5]}")
else:
    print("TabSyn quality check PASSED -- ready for assembly")

write_status(f"QUALITY CHECK DONE | AUC={adv['adversarial_auc']}")